In [42]:
import sqlite3

# 1. เชื่อมต่อ/สร้างไฟล์ Database
conn = sqlite3.connect('set_stocks.db')
cursor = conn.cursor()

# 2. สร้าง Tables ตามหลัก Normalization
cursor.executescript('''
    -- ตารางเก็บรายชื่อหุ้น
    CREATE TABLE IF NOT EXISTS Stocks (
        stock_id INTEGER PRIMARY KEY AUTOINCREMENT,
        symbol TEXT UNIQUE NOT NULL
    );

    -- ตารางเก็บหมวดหมู่
    CREATE TABLE IF NOT EXISTS Categories (
        cat_id INTEGER PRIMARY KEY AUTOINCREMENT,
        cat_name TEXT UNIQUE NOT NULL
    );

    -- ตารางเชื่อมโยง (Mapping) - หุ้น 1 ตัวอยู่ได้หลาย Cat
    CREATE TABLE IF NOT EXISTS Stock_Category_Mapping (
        stock_id INTEGER,
        cat_id INTEGER,
        FOREIGN KEY (stock_id) REFERENCES Stocks(stock_id),
        FOREIGN KEY (cat_id) REFERENCES Categories(cat_id),
        PRIMARY KEY (stock_id, cat_id)
    );
''')

# 3. ข้อมูลสมมติ (ในอนาคตคุณสามารถดึงจาก API มาใส่ตรงนี้ได้)
raw_data = [
    {'symbol': 'PTT', 'cats': ['SET50', 'ENERG']},
    {'symbol': 'CPALL', 'cats': ['SET50', 'COMM']},
    {'symbol': 'AOT', 'cats': ['SET50', 'TRANS']},
    {'symbol': 'TMB', 'cats': ['SET100', 'BANK']}
]

# 4. ฟังก์ชันยัดข้อมูลลงตารางแบบไม่ซ้ำ (Normalization Process)
for item in raw_data:
    # Insert หุ้น (ถ้ามีแล้วจะข้าม)
    cursor.execute("INSERT OR IGNORE INTO Stocks (symbol) VALUES (?)", (item['symbol'],))
    cursor.execute("SELECT stock_id FROM Stocks WHERE symbol = ?", (item['symbol'],))
    s_id = cursor.fetchone()[0]

    for cat in item['cats']:
        # Insert หมวดหมู่ (ถ้ามีแล้วจะข้าม)
        cursor.execute("INSERT OR IGNORE INTO Categories (cat_name) VALUES (?)", (cat,))
        cursor.execute("SELECT cat_id FROM Categories WHERE cat_name = ?", (cat,))
        c_id = cursor.fetchone()[0]

        # เชื่อม Key เข้าด้วยกันในตาราง Mapping
        cursor.execute("INSERT OR IGNORE INTO Stock_Category_Mapping (stock_id, cat_id) VALUES (?, ?)", (s_id, c_id))

conn.commit()

# --- ส่วนของการตอบโจทย์ 3 ข้อ ---

print("--- RESULTS ---")

# ข้อ 1: How many stock categories?
cursor.execute("SELECT COUNT(*) FROM Categories")
print(f"1. Total Categories: {cursor.fetchone()[0]}")

# ข้อ 2: List of categories?
cursor.execute("SELECT cat_name FROM Categories")
cats = [row[0] for row in cursor.fetchall()]
print(f"2. All Categories: {', '.join(cats)}")

# ข้อ 3: List stock in each category? (ตัวอย่าง: เจาะจงดู SET50)
target_cat = 'SET50'
query = '''
    SELECT s.symbol 
    FROM Stocks s
    JOIN Stock_Category_Mapping scm ON s.stock_id = scm.stock_id
    JOIN Categories c ON scm.cat_id = c.cat_id
    WHERE c.cat_name = ?
'''
cursor.execute(query, (target_cat,))
stocks_in_cat = [row[0] for row in cursor.fetchall()]
print(f"3. Stocks in {target_cat}: {', '.join(stocks_in_cat)}")

conn.close()

--- RESULTS ---
1. Total Categories: 6
2. All Categories: SET50, ENERG, COMM, TRANS, SET100, BANK
3. Stocks in SET50: PTT, CPALL, AOT


In [44]:
import sqlite3

# สร้างการเชื่อมต่อ Database
conn = sqlite3.connect('stock_market.db')
cursor = conn.cursor()

# 1. สร้างตาราง Categories (ถ้ายังไม่มี)
cursor.execute('''
    CREATE TABLE IF NOT EXISTS Categories (
        cat_id INTEGER PRIMARY KEY AUTOINCREMENT,
        cat_name TEXT UNIQUE NOT NULL
    )
''')

# 2. เพิ่มข้อมูลหมวดหมู่ตัวอย่างลงไป (Assume ว่าเราดึงข้อมูลจาก SET มาแล้ว)
sample_cats = [('SET50',), ('SET100',), ('ENERG',), ('BANK',), ('COMM',), ('TRANS',)]
cursor.executemany("INSERT OR IGNORE INTO Categories (cat_name) VALUES (?)", sample_cats)
conn.commit()

print("--- Database is ready ---")

--- Database is ready ---


In [46]:
# โค้ดสำหรับคำถามข้อที่ 1
cursor.execute("SELECT COUNT(*) FROM Categories")
total_categories = cursor.fetchone()[0]

print(f"Question: How many stock categories?")
print(f"Answer: There are {total_categories} categories in the database.")

# ปิดการเชื่อมต่อ
conn.close()

Question: How many stock categories?
Answer: There are 6 categories in the database.


In [48]:
-- 1. สร้างตารางหมวดหมู่ (ตามหลัก Normalization)
CREATE TABLE IF NOT EXISTS Categories (
    cat_id INTEGER PRIMARY KEY AUTOINCREMENT,
    cat_name TEXT UNIQUE NOT NULL
);

-- 2. ตัวอย่างการเพิ่มข้อมูล (ใส่กี่ครั้งก็ได้ ถ้าซ้ำระบบจะ Ignore เพราะเราตั้ง UNIQUE ไว้)
INSERT OR IGNORE INTO Categories (cat_name) VALUES ('SET50');
INSERT OR IGNORE INTO Categories (cat_name) VALUES ('SET100');
INSERT OR IGNORE INTO Categories (cat_name) VALUES ('ENERG');
INSERT OR IGNORE INTO Categories (cat_name) VALUES ('BANK');
INSERT OR IGNORE INTO Categories (cat_name) VALUES ('COMM');
INSERT OR IGNORE INTO Categories (cat_name) VALUES ('TRANS');

-- 3. SQL สำหรับตอบคำถาม: How many stock categories?
SELECT COUNT(*) AS total_categories FROM Categories;

SyntaxError: invalid syntax (1236495422.py, line 1)

In [50]:
import sqlite3

# 1. เชื่อมต่อฐานข้อมูล (สร้างไฟล์ชื่อ stock_data.db)
conn = sqlite3.connect('stock_data.db')
cursor = conn.cursor()

# 2. นิยาม Schema (ตาราง Categories)
# เราใช้ ''' เพื่อครอบ SQL หลายบรรทัด
create_table_sql = '''
CREATE TABLE IF NOT EXISTS Categories (
    cat_id INTEGER PRIMARY KEY AUTOINCREMENT,
    cat_name TEXT UNIQUE NOT NULL
);
'''

# 3. รันคำสั่งสร้างตาราง
cursor.execute(create_table_sql)

# 4. ใส่ข้อมูลหมวดหมู่ (ตัวอย่าง 6 หมวด)
categories = [('SET50',), ('SET100',), ('ENERG',), ('BANK',), ('COMM',), ('TRANS',)]
cursor.executemany("INSERT OR IGNORE INTO Categories (cat_name) VALUES (?)", categories)
conn.commit()

# 5. SQL สำหรับตอบคำถาม: How many stock categories?
cursor.execute("SELECT COUNT(*) FROM Categories")
result = cursor.fetchone()[0]

print(f"--- Question 1 ---")
print(f"How many stock categories?")
print(f"Answer: {result}")

# ปิดการเชื่อมต่อ
conn.close()

--- Question 1 ---
How many stock categories?
Answer: 6


In [52]:
import sqlite3
import pandas as pd

# 1. เชื่อมต่อฐานข้อมูล
conn = sqlite3.connect('stock_data.db')

# 2. ใช้ Pandas อ่านคำสั่ง SQL เพื่อดึงจำนวน Category
# เราจะตั้งชื่อ Column ว่า 'total_categories' เพื่อให้หัวตารางดูสวยงาม
query = "SELECT COUNT(*) AS total_categories FROM Categories"
df_result = pd.read_sql_query(query, conn)

# 3. แสดงผลในรูปแบบตาราง DataFrame
# ใน Jupyter Notebook หรือ Colab การพิมพ์ชื่อตัวแปรเฉยๆ จะแสดงผลเป็นตารางสวยงาม
display(df_result)

# ปิดการเชื่อมต่อ
conn.close()

,total_categories
0,6


In [56]:
import psycopg2
from settrade_v2 import Investor
import pandas as pd

# --- STEP 1: เชื่อมต่อเพื่อสร้าง Database (ถ้ายังไม่มี) ---
try:
    # เชื่อมต่อกับ db 'postgres' ซึ่งเป็น db พื้นฐานที่มีทุกเครื่อง
    temp_conn = psycopg2.connect(
        host="127.0.0.1",
        user="postgres",
        password="Shifa.326459" # ใส่รหัสผ่านของคุณ
    )
    temp_conn.autocommit = True
    temp_cur = temp_conn.cursor()
    
    # ตรวจสอบว่ามี db นี้อยู่หรือยัง ถ้าไม่มีให้สร้าง
    temp_cur.execute("SELECT 1 FROM pg_catalog.pg_database WHERE datname = 'thai_stocks_db';")
    exists = temp_cur.fetchone()
    if not exists:
        temp_cur.execute("CREATE DATABASE thai_stocks_db;")
        print("✅ Created database: thai_stocks_db")
    
    temp_cur.close()
    temp_conn.close()

except Exception as e:
    print(f"❌ Error during DB creation: {e}")

# --- STEP 2: เชื่อมต่อเข้าสู่ thai_stocks_db เพื่อสร้าง Table ---
conn = psycopg2.connect(
    host="127.0.0.1",
    database="thai_stocks_db",
    user="postgres",
    password="672437002"
)
conn.autocommit = True
cur = conn.cursor()

print("🏗️ Creating Normalized Tables...")
cur.execute("DROP TABLE IF EXISTS stocks CASCADE;")
cur.execute("DROP TABLE IF EXISTS sectors CASCADE;")

# 1. ตาราง Sectors (Category)
cur.execute("""
    CREATE TABLE sectors (
        sector_id SERIAL PRIMARY KEY,
        sector_name VARCHAR(100) UNIQUE NOT NULL
    );
""")

# 2. ตาราง Stocks (เชื่อมกับ Sector ด้วย ID)
cur.execute("""
    CREATE TABLE stocks (
        symbol VARCHAR(20) PRIMARY KEY,
        sector_id INTEGER REFERENCES sectors(sector_id)
    );
""")

# --- STEP 3: ดึงข้อมูลจาก API และบันทึกแบบเชื่อมความสัมพันธ์ ---
investor = Investor(
    app_id="m6P7C2ceQ5ffMv7k", 
    app_secret="AIlXY/4yHMpjZN5WthcCGDSGSyL8XWsh0WdyAnFWbzxo", 
    broker_id="SANDBOX",
    app_code="SANDBOX"
)
market = investor.MarketData()

# รายชื่อหุ้นตัวอย่าง
target_symbols = ["ADVANC", "AOT", "KBANK", "CPALL", "PTT", "SCB", "BDMS"]

for symbol in target_symbols:
    try:
        info = market.get_quote_symbol(symbol)
        sec_name = info.get('sector', 'SET100')
        
        # ใส่ชื่อกลุ่มในตาราง sectors ก่อน
        cur.execute("INSERT INTO sectors (sector_name) VALUES (%s) ON CONFLICT DO NOTHING", (sec_name,))
        
        # ดึง ID ของกลุ่มนั้นมาเพื่อใช้เป็น Foreign Key
        cur.execute("SELECT sector_id FROM sectors WHERE sector_name = %s", (sec_name,))
        s_id = cur.fetchone()[0]
        
        # ใส่ชื่อหุ้นและเชื่อม ID ของกลุ่ม
        cur.execute("INSERT INTO stocks (symbol, sector_id) VALUES (%s, %s) ON CONFLICT DO NOTHING", (symbol, s_id))
        print(f"💾 Saved: {symbol} (Category ID: {s_id})")
        
    except Exception as e:
        print(f"⚠️ Error {symbol}: {e}")

# --- STEP 4: ตรวจสอบผลลัพธ์ ---
print("\n--- 📊 Summary ---")
# นับจำนวน Categories
cur.execute("SELECT COUNT(*) FROM sectors;")
print(f"How many stock categories?: {cur.fetchone()[0]}")

# รายชื่อหุ้นแยกตามกลุ่ม (ใช้ JOIN)
query = """
    SELECT s.sector_name AS "Category", st.symbol AS "Stock"
    FROM sectors s
    JOIN stocks st ON s.sector_id = st.sector_id
    ORDER BY s.sector_name;
"""
df = pd.read_sql(query, conn)
display(df)

✅ Created database: thai_stocks_db
🏗️ Creating Normalized Tables...
💾 Saved: ADVANC (Category ID: 1)
💾 Saved: AOT (Category ID: 1)
💾 Saved: KBANK (Category ID: 1)
💾 Saved: CPALL (Category ID: 1)
💾 Saved: PTT (Category ID: 1)
💾 Saved: SCB (Category ID: 1)
💾 Saved: BDMS (Category ID: 1)

--- 📊 Summary ---
How many stock categories?: 1


/var/folders/s1/d92l8pcd3xj0465l248s42800000gp/T/ipykernel_53125/2044022015.py:103: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,Category,Stock
0,SET100,ADVANC
1,SET100,AOT
2,SET100,KBANK
3,SET100,CPALL
4,SET100,PTT
5,SET100,SCB
6,SET100,BDMS


In [60]:
import pandas as pd
from IPython.display import display, Markdown

# ฟังก์ชันช่วยแสดงหัวข้อ
def print_header(text):
    display(Markdown(f"### 📋 {text}"))

# 1. คำถาม: List of categories? (รายชื่อกลุ่มอุตสาหกรรมทั้งหมด)
print_header("รายชื่อกลุ่มอุตสาหกรรม (List of Categories)")

# ใช้ SQL ดึงเฉพาะชื่อกลุ่มจากตาราง sectors และเรียงลำดับตามตัวอักษร
sql_categories = """
    SELECT 
        sector_id AS "ID", 
        sector_name AS "Industry Category (ชื่อกลุ่ม)" 
    FROM sectors 
    ORDER BY sector_name ASC;
"""

# ใช้ Pandas อ่านข้อมูลออกมาแสดงเป็นตารางสวยๆ
df_categories = pd.read_sql(sql_categories, conn)

# แสดงผล
display(df_categories)

# 2. แถมเพิ่มเติม: นับจำนวนหุ้นในแต่ละกลุ่ม (เพื่อให้เห็นภาพการเชื่อมโยง Foreign Key)
print_header("สรุปจำนวนหุ้นในแต่ละกลุ่ม (Stocks Count per Category)")

sql_count_per_cat = """
    SELECT 
        s.sector_name AS "Category",
        COUNT(st.symbol) AS "Number of Stocks (จำนวนหุ้น)"
    FROM sectors s
    LEFT JOIN stocks st ON s.sector_id = st.sector_id
    GROUP BY s.sector_name
    ORDER BY "Number of Stocks (จำนวนหุ้น)" DESC;
"""

df_count = pd.read_sql(sql_count_per_cat, conn)
display(df_count)

### 📋 รายชื่อกลุ่มอุตสาหกรรม (List of Categories)

/var/folders/s1/d92l8pcd3xj0465l248s42800000gp/T/ipykernel_53125/1120301146.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_categories = pd.read_sql(sql_categories, conn)


,ID,Industry Category (ชื่อกลุ่ม)
0,1,SET100


### 📋 สรุปจำนวนหุ้นในแต่ละกลุ่ม (Stocks Count per Category)

/var/folders/s1/d92l8pcd3xj0465l248s42800000gp/T/ipykernel_53125/1120301146.py:39: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_count = pd.read_sql(sql_count_per_cat, conn)


,Category,Number of Stocks (จำนวนหุ้น)
0,SET100,7


In [62]:
import pandas as pd
from IPython.display import display, Markdown

# ฟังก์ชันแสดงหัวข้อ
def print_header(text):
    display(Markdown(f"### 🔍 {text}"))

# 1. คำถาม: List all stocks in each category? (รายชื่อหุ้นแยกตามกลุ่ม)
print_header("เจาะลึก: รายชื่อหุ้นในแต่ละกลุ่มอุตสาหกรรม")

# SQL Query: เชื่อม 2 ตารางเข้าด้วยกันด้วย JOIN
# และใช้ STRING_AGG เพื่อรวมชื่อหุ้นคั่นด้วยเครื่องหมายจุลภาค (,)
sql_stocks_in_cat = """
    SELECT 
        s.sector_name AS "Category (กลุ่ม)",
        COUNT(st.symbol) AS "Count (จำนวน)",
        STRING_AGG(st.symbol, ', ' ORDER BY st.symbol) AS "Stock List (รายชื่อหุ้น)"
    FROM sectors s
    LEFT JOIN stocks st ON s.sector_id = st.sector_id
    GROUP BY s.sector_name
    ORDER BY "Count (จำนวน)" DESC;
"""

# ดึงข้อมูลมาแสดงผลด้วย Pandas
df_detailed = pd.read_sql(sql_stocks_in_cat, conn)

# ตั้งค่าให้ Pandas แสดงข้อความในคอลัมน์ได้ยาวๆ ไม่ตัดคำ (...)
pd.set_option('display.max_colwidth', None)

display(df_detailed)

### 🔍 เจาะลึก: รายชื่อหุ้นในแต่ละกลุ่มอุตสาหกรรม

/var/folders/s1/d92l8pcd3xj0465l248s42800000gp/T/ipykernel_53125/1794967755.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_detailed = pd.read_sql(sql_stocks_in_cat, conn)


,Category (กลุ่ม),Count (จำนวน),Stock List (รายชื่อหุ้น)
0,SET100,7,"ADVANC, AOT, BDMS, CPALL, KBANK, PTT, SCB"


In [64]:
import psycopg2
import time
from settrade_v2 import Investor
import pandas as pd
from IPython.display import display

# 1. เชื่อมต่อ Database (ตรวจสอบรหัสผ่านของคุณอีกครั้ง)
conn = psycopg2.connect(
    host="127.0.0.1",
    database="thai_stocks_db",
    user="postgres",
    password="672437002" 
)
conn.autocommit = True
cur = conn.cursor()

# 2. รายชื่อหุ้น 30 ตัวจาก SET100 (Big Cap Samples)
set100_list = [
    "ADVANC", "AOT", "AWC", "BANPU", "BBL", "BDMS", "BEM", "BGRIM", "BH", "BTS",
    "CBG", "CPALL", "CPF", "CPN", "CRC", "DELTA", "EA", "EGCO", "GLOBAL", "GPSC",
    "GULF", "HMPRO", "INTUCH", "IVL", "KBANK", "KCE", "KTB", "KTC", "LH", "MINT"
]

# 3. เชื่อมต่อ API Settrade
investor = Investor(
    app_id="m6P7C2ceQ5ffMv7k", 
    app_secret="AIlXY/4yHMpjZN5WthcCGDSGSyL8XWsh0WdyAnFWbzxo", 
    broker_id="SANDBOX",
    app_code="SANDBOX"
)
market = investor.MarketData()

print("🚀 เริ่มกระบวนการดึงข้อมูลหุ้น 30 ตัวและบันทึกแบบ Normalized...")

for symbol in set100_list:
    try:
        # ดึงข้อมูลเพื่อหา Sector จริงจากตลาด
        info = market.get_quote_symbol(symbol)
        real_sector = info.get('sector', 'ETC') 
        
        # --- ขั้นตอนการบันทึกแบบแยกตาราง (Normalization) ---
        
        # A. บันทึกชื่อกลุ่มลงตาราง sectors ก่อน (ถ้ามีอยู่แล้วจะไม่ทำอะไร)
        cur.execute("""
            INSERT INTO sectors (sector_name) 
            VALUES (%s) 
            ON CONFLICT (sector_name) DO NOTHING;
        """, (real_sector,))
        
        # B. ดึง ID ของกลุ่มนั้นออกมาใช้เป็น Foreign Key
        cur.execute("SELECT sector_id FROM sectors WHERE sector_name = %s", (real_sector,))
        s_id = cur.fetchone()[0]
        
        # C. บันทึกหุ้นลงตาราง stocks โดยผูกกับ ID ของกลุ่ม (ประหยัดพื้นที่และลดความซ้ำซ้อน)
        cur.execute("""
            INSERT INTO stocks (symbol, sector_id) 
            VALUES (%s, %s) 
            ON CONFLICT (symbol) DO NOTHING;
        """, (symbol, s_id))
        
        print(f"✅ บันทึกสำเร็จ: {symbol:7} | กลุ่ม: {real_sector}")
        
        # หน่วงเวลาเล็กน้อยเพื่อความเสถียรของ API
        time.sleep(0.1)

    except Exception as e:
        print(f"⚠️ เกิดข้อผิดพลาดกับหุ้น {symbol}: {e}")

# 4. แสดงผลลัพธ์ด้วยการ JOIN ตาราง (เพื่อดูว่าเชื่อมกันถูกต้องไหม)
print("\n" + "="*60)
print("📊 สรุปข้อมูลใน Database (เชื่อมตาราง Sectors และ Stocks)")
print("="*60)

sql_join = """
    SELECT 
        s.sector_name AS "Industry Group",
        COUNT(st.symbol) AS "Total Stocks",
        STRING_AGG(st.symbol, ', ') AS "Symbols"
    FROM sectors s
    JOIN stocks st ON s.sector_id = st.sector_id
    GROUP BY s.sector_name
    ORDER BY "Total Stocks" DESC;
"""

df_final = pd.read_sql(sql_join, conn)
pd.set_option('display.max_colwidth', None)
display(df_final)

# สรุปภาพรวม
cur.execute("SELECT COUNT(*) FROM stocks;")
total_s = cur.fetchone()[0]
print(f"\n✨ เรียบร้อย! ขณะนี้มีหุ้นทั้งหมด {total_s} ตัวในฐานข้อมูลของคุณ")

🚀 เริ่มกระบวนการดึงข้อมูลหุ้น 30 ตัวและบันทึกแบบ Normalized...
✅ บันทึกสำเร็จ: ADVANC  | กลุ่ม: ETC
✅ บันทึกสำเร็จ: AOT     | กลุ่ม: ETC
✅ บันทึกสำเร็จ: AWC     | กลุ่ม: ETC
✅ บันทึกสำเร็จ: BANPU   | กลุ่ม: ETC
✅ บันทึกสำเร็จ: BBL     | กลุ่ม: ETC
✅ บันทึกสำเร็จ: BDMS    | กลุ่ม: ETC
✅ บันทึกสำเร็จ: BEM     | กลุ่ม: ETC
✅ บันทึกสำเร็จ: BGRIM   | กลุ่ม: ETC
✅ บันทึกสำเร็จ: BH      | กลุ่ม: ETC
✅ บันทึกสำเร็จ: BTS     | กลุ่ม: ETC
✅ บันทึกสำเร็จ: CBG     | กลุ่ม: ETC
✅ บันทึกสำเร็จ: CPALL   | กลุ่ม: ETC
✅ บันทึกสำเร็จ: CPF     | กลุ่ม: ETC
✅ บันทึกสำเร็จ: CPN     | กลุ่ม: ETC
✅ บันทึกสำเร็จ: CRC     | กลุ่ม: ETC
✅ บันทึกสำเร็จ: DELTA   | กลุ่ม: ETC
✅ บันทึกสำเร็จ: EA      | กลุ่ม: ETC
✅ บันทึกสำเร็จ: EGCO    | กลุ่ม: ETC
✅ บันทึกสำเร็จ: GLOBAL  | กลุ่ม: ETC
✅ บันทึกสำเร็จ: GPSC    | กลุ่ม: ETC
✅ บันทึกสำเร็จ: GULF    | กลุ่ม: ETC
✅ บันทึกสำเร็จ: HMPRO   | กลุ่ม: ETC
⚠️ เกิดข้อผิดพลาดกับหุ้น INTUCH: Symbol not found
✅ บันทึกสำเร็จ: IVL     | กลุ่ม: ETC
✅ บันทึกสำเร็จ: KBANK   | กลุ่ม: ETC

/var/folders/s1/d92l8pcd3xj0465l248s42800000gp/T/ipykernel_53125/606515618.py:85: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_final = pd.read_sql(sql_join, conn)


,Industry Group,Total Stocks,Symbols
0,ETC,24,"AWC, BANPU, BBL, BEM, BGRIM, BH, BTS, CBG, CPF, CPN, CRC, DELTA, EA, EGCO, GLOBAL, GPSC, GULF, HMPRO, IVL, KCE, KTB, KTC, LH, MINT"
1,SET100,7,"ADVANC, AOT, KBANK, CPALL, PTT, SCB, BDMS"



✨ เรียบร้อย! ขณะนี้มีหุ้นทั้งหมด 31 ตัวในฐานข้อมูลของคุณ


In [66]:
import pandas as pd
from IPython.display import display, Markdown

# ฟังก์ชันช่วยแสดงหัวข้อให้สวยงาม
def print_header(text):
    display(Markdown(f"### 🔍 {text}"))

# --- คำถามที่ 3: List stock in each categories? ---
print_header("รายชื่อหุ้นแยกตามกลุ่มอุตสาหกรรม (Detailed Stocks per Category)")

# SQL Query: เชื่อมตาราง sectors และ stocks เข้าด้วยกัน
# เราใช้ LEFT JOIN เพื่อให้แสดงทุก Category แม้บางกลุ่มจะยังไม่มีหุ้นก็ตาม
sql_list_each_cat = """
    SELECT 
        s.sector_name AS "Category (กลุ่มอุตสาหกรรม)",
        COUNT(st.symbol) AS "Count (จำนวนหุ้น)",
        STRING_AGG(st.symbol, ', ' ORDER BY st.symbol) AS "Stock List (รายชื่อหุ้น)"
    FROM sectors s
    LEFT JOIN stocks st ON s.sector_id = st.sector_id
    GROUP BY s.sector_name
    ORDER BY "Count (จำนวนหุ้น)" DESC;
"""

# ดึงข้อมูลมาแสดงผลด้วย Pandas
df_list_cat = pd.read_sql(sql_list_each_cat, conn)

# ตั้งค่า Pandas ให้แสดงข้อความยาวๆ ได้ (ไม่ตัดเป็น ...)
pd.set_option('display.max_colwidth', None)

display(df_list_cat)

# แถม: สรุปภาพรวมสั้นๆ
cur.execute("SELECT COUNT(*) FROM stocks;")
total_stocks = cur.fetchone()[0]
print(f"\n📊 รวมทั้งหมดมีหุ้น {total_stocks} ตัว กระจายอยู่ในกลุ่มต่างๆ ตามตารางด้านบน")

### 🔍 รายชื่อหุ้นแยกตามกลุ่มอุตสาหกรรม (Detailed Stocks per Category)

/var/folders/s1/d92l8pcd3xj0465l248s42800000gp/T/ipykernel_53125/3948593023.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_list_cat = pd.read_sql(sql_list_each_cat, conn)


,Category (กลุ่มอุตสาหกรรม),Count (จำนวนหุ้น),Stock List (รายชื่อหุ้น)
0,ETC,24,"AWC, BANPU, BBL, BEM, BGRIM, BH, BTS, CBG, CPF, CPN, CRC, DELTA, EA, EGCO, GLOBAL, GPSC, GULF, HMPRO, IVL, KCE, KTB, KTC, LH, MINT"
1,SET100,7,"ADVANC, AOT, BDMS, CPALL, KBANK, PTT, SCB"



📊 รวมทั้งหมดมีหุ้น 31 ตัว กระจายอยู่ในกลุ่มต่างๆ ตามตารางด้านบน


In [68]:
import pandas as pd
from IPython.display import display, Markdown

# ฟังก์ชันช่วยแสดงหัวข้อ
def print_header(text):
    display(Markdown(f"### 📋 {text}"))

# --- คำถามสุดท้าย: List all stock names? ---
print_header("รายชื่อหุ้นทั้งหมด (All Stock Names)")

# SQL Query: ดึงเฉพาะชื่อหุ้นจากตาราง stocks
# เรียงลำดับตามตัวอักษร (A-Z)
sql_all_stocks = """
    SELECT 
        symbol AS "Stock Symbol (ชื่อย่อหุ้น)"
    FROM stocks 
    ORDER BY symbol ASC;
"""

# ใช้ Pandas ดึงข้อมูลมาแสดงผล
df_all_stocks = pd.read_sql(sql_all_stocks, conn)

# แสดงผลรายชื่อหุ้นทั้งหมด
display(df_all_stocks)

# แถม: สรุปเป็นข้อความบรรทัดเดียว (เผื่อเอาไปใช้ Copy)
cur.execute("SELECT symbol FROM stocks ORDER BY symbol;")
all_list = [row[0] for row in cur.fetchall()]
print(f"\n🔗 รวมรายชื่อหุ้นทั้งหมด ({len(all_list)} ตัว):")
print(", ".join(all_list))

### 📋 รายชื่อหุ้นทั้งหมด (All Stock Names)

/var/folders/s1/d92l8pcd3xj0465l248s42800000gp/T/ipykernel_53125/3092979735.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_all_stocks = pd.read_sql(sql_all_stocks, conn)


,Stock Symbol (ชื่อย่อหุ้น)
0,ADVANC
1,AOT
2,AWC
3,BANPU
4,BBL
5,BDMS
6,BEM
7,BGRIM
8,BH
9,BTS



🔗 รวมรายชื่อหุ้นทั้งหมด (31 ตัว):
ADVANC, AOT, AWC, BANPU, BBL, BDMS, BEM, BGRIM, BH, BTS, CBG, CPALL, CPF, CPN, CRC, DELTA, EA, EGCO, GLOBAL, GPSC, GULF, HMPRO, IVL, KBANK, KCE, KTB, KTC, LH, MINT, PTT, SCB


In [74]:
import pandas as pd
import psycopg2
from IPython.display import display, Markdown

# 1. เชื่อมต่อ Database (ใช้ข้อมูลของคุณ)
conn = psycopg2.connect(
    host="127.0.0.1",
    database="thai_stocks_db",
    user="postgres",
    password="672437002"
)

# ฟังก์ชันช่วยแสดงหัวข้อแบบสวยงาม
def print_section(title):
    display(Markdown(f"--- \n## 📊 {title}"))

# =========================================================
# 5.1 How many stocks? (จำนวนหุ้นทั้งหมด)
# =========================================================
print_section("5.1 How many stocks?")
sql_51 = "SELECT COUNT(*) AS \"Total Stocks (จำนวนหุ้นทั้งหมด)\" FROM stocks;"
df_51 = pd.read_sql(sql_51, conn)
display(df_51)

# =========================================================
# 5.2 List all stock name? (รายชื่อหุ้นทั้งหมด)
# =========================================================
print_section("5.2 List all stock name?")
sql_52 = "SELECT symbol AS \"Symbol (ชื่อย่อหุ้น)\" FROM stocks ORDER BY symbol ASC;"
df_52 = pd.read_sql(sql_52, conn)
display(df_52)

# =========================================================
# 5.3 How many stock categories? (จำนวนกลุ่มอุตสาหกรรม)
# =========================================================
print_section("5.3 How many stock categories?")
sql_53 = "SELECT COUNT(*) AS \"Total Categories (จำนวนกลุ่ม)\" FROM sectors;"
df_53 = pd.read_sql(sql_53, conn)
display(df_53)

# =========================================================
# 5.4 List of categories? (รายชื่อกลุ่มอุตสาหกรรม)
# =========================================================
print_section("5.4 List of categories?")
sql_54 = "SELECT sector_id AS \"ID\", sector_name AS \"Category Name\" FROM sectors ORDER BY sector_name;"
df_54 = pd.read_sql(sql_54, conn)
display(df_54)

# =========================================================
# 5.5 List stock in each categories? (รายชื่อหุ้นแยกตามกลุ่ม)
# =========================================================
print_section("5.5 List stock in each categories?")
# ใช้ JOIN เพื่อเชื่อมข้อมูล และ STRING_AGG เพื่อรวมรายชื่อหุ้น
sql_55 = """
    SELECT 
        s.sector_name AS "Category",
        COUNT(st.symbol) AS "Count",
        STRING_AGG(st.symbol, ', ' ORDER BY st.symbol) AS "Stock List"
    FROM sectors s
    LEFT JOIN stocks st ON s.sector_id = st.sector_id
    GROUP BY s.sector_name
    ORDER BY "Count" DESC;
"""
# ตั้งค่าให้ Pandas ไม่ตัดคำยาวๆ
pd.set_option('display.max_colwidth', None)
df_55 = pd.read_sql(sql_55, conn)
display(df_55)

print("\n✅ การวิเคราะห์เชิงปริมาณ (Quantitative Analysis) เสร็จสิ้นเรียบร้อยแล้ว!")

--- 
## 📊 5.1 How many stocks?

/var/folders/s1/d92l8pcd3xj0465l248s42800000gp/T/ipykernel_53125/1070719697.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_51 = pd.read_sql(sql_51, conn)


,Total Stocks (จำนวนหุ้นทั้งหมด)
0,31


--- 
## 📊 5.2 List all stock name?

/var/folders/s1/d92l8pcd3xj0465l248s42800000gp/T/ipykernel_53125/1070719697.py:30: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_52 = pd.read_sql(sql_52, conn)


...

,ID,Category Name
0,8,ETC
1,1,SET100


--- 
## 📊 5.5 List stock in each categories?

/var/folders/s1/d92l8pcd3xj0465l248s42800000gp/T/ipykernel_53125/1070719697.py:66: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_55 = pd.read_sql(sql_55, conn)


,Category,Count,Stock List
0,ETC,24,"AWC, BANPU, BBL, BEM, BGRIM, BH, BTS, CBG, CPF, CPN, CRC, DELTA, EA, EGCO, GLOBAL, GPSC, GULF, HMPRO, IVL, KCE, KTB, KTC, LH, MINT"
1,SET100,7,"ADVANC, AOT, BDMS, CPALL, KBANK, PTT, SCB"



✅ การวิเคราะห์เชิงปริมาณ (Quantitative Analysis) เสร็จสิ้นเรียบร้อยแล้ว!


In [ ]:
import time

# 1. เพิ่มคอลัมน์ market_cap ในตาราง stocks (ถ้ายังไม่มี)
cur.execute("ALTER TABLE stocks ADD COLUMN IF NOT EXISTS market_cap NUMERIC;")

print("🔄 กำลังอัปเดตข้อมูล Market Cap จาก API...")
cur.execute("SELECT symbol FROM stocks;")
all_symbols = [row[0] for row in cur.fetchall()]

for symbol in all_symbols:
    try:
        info = market.get_quote_symbol(symbol)
        mkt_cap = info.get('marketCap', 0)
        
        # อัปเดตข้อมูลลง Database
        cur.execute("UPDATE stocks SET market_cap = %s WHERE symbol = %s", (mkt_cap, symbol))
        print(f"✅ Updated {symbol}: {mkt_cap:,.2f}")
        time.sleep(0.1)
    except Exception as e:
        print(f"⚠️ Error {symbol}: {e}")

print("--- อัปเดตข้อมูล Market Cap เสร็จสมบูรณ์ ---")